# 11 - Control Logic: Overfit, Checkpoint, Curriculum

This notebook isolates the orchestration helpers in `control.py` that the
trainer delegates to: `OverfitManager` (collapsing a loader to one repeated
batch and deciding when to stop), `Checkpoint` (tracking the best validation
loss), and `CurriculumController` (swapping the warmup loss config for the
complete one at a configured epoch and resetting schedules). We exercise each
in isolation and visualise their decisions.

Modules exercised: `OverfitManager`, `Checkpoint`, `CurriculumController` from
`pipelines.backbone_pipeline.control`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
import copy
from configuration.training_config import (
    TrainerConfig, GaussianConfig, OverfitConfig, LossConfig, LossCurriculumConfig,
)
from pipelines.backbone_pipeline.control   import OverfitManager, Checkpoint, CurriculumController
from pipelines.backbone_pipeline.loss      import Loss
from pipelines.backbone_pipeline.callbacks import Warmup, Scheduler, EarlyStopping
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

gaussian_cfg = GaussianConfig(n_default_gaussians=2, x_min=-10.0, x_max=40.0)
base_cfg     = TrainerConfig(gaussian=gaussian_cfg)
nl, nt       = NullLogger(), NullTracker()


## OverfitManager loader collapse

Given a multi-batch loader, `setup_loaders` extracts the first batch, trims it
to `batch_size`, and returns a synthetic loader of that single batch repeated
`min(len(loader), max_steps)` times. We confirm the collapse and the trimmed
batch dimension.

In [ ]:
cfg = copy.deepcopy(base_cfg)
cfg.overfit = OverfitConfig(enabled=True, max_steps=6, stop_threshold=1e-6, batch_size=2)
om = OverfitManager(cfg, nl)

torch.manual_seed(SEED)
full_loader = [(torch.randn(8, 4, 8, 8), torch.randn(8, 6, 8, 8)) for _ in range(10)]
train_c, val_c, test_c = om.setup_loaders(full_loader, full_loader, full_loader)

print('original loader length :', len(full_loader))
print('collapsed train length :', len(train_c), '(== min(len, max_steps))')
print('collapsed batch shape  :', tuple(train_c[0][0].shape), '(batch trimmed to', cfg.overfit.batch_size, ')')


## Checkpoint best-loss tracking

`Checkpoint.step` records a new best only when the validation loss improves.
We feed a noisy decreasing validation sequence (saving disabled by using a
stub trainer) and plot the running best against the raw values.

In [ ]:
class StubTrainer:
    def capture_state(self, epoch):
        return {}

ckpt = Checkpoint(nl, nt, save_path='/tmp/nb11_unused.pt')
ckpt.save = lambda trainer, path, epoch: None

torch.manual_seed(SEED)
val_seq = (1.0 - np.linspace(0, 0.6, 40)) + 0.08 * np.random.randn(40)
best_track, improved = [], []
stub = StubTrainer()
for e, v in enumerate(val_seq):
    is_best = ckpt.step(float(v), e, stub)
    best_track.append(ckpt.best_val_loss)
    improved.append(is_best)

improved = np.array(improved)
epochs = np.arange(len(val_seq))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, val_seq, color='lightgrey', marker='.', label='val loss')
ax.plot(epochs, best_track, color='C0', label='running best')
ax.scatter(epochs[improved], val_seq[improved], color='C2', zorder=5, label='new best')
ax.set_xlabel('epoch'); ax.set_ylabel('val loss')
ax.set_title('Checkpoint best-loss tracking')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print('final best loss : %.4f at epoch %d' % (ckpt.best_val_loss, ckpt.best_epoch))


## Curriculum loss swap

`CurriculumController.maybe_swap` replaces the active loss configuration with
the complete-stage config at `swap_epoch`. We build a warmup config that uses
only `param_l1` and a complete config that adds `mse_curve`, then confirm the
set of active loss terms changes exactly at the swap epoch.

In [ ]:
x_axis = torch.linspace(-10.0, 40.0, 24, dtype=torch.float32)

warmup_loss   = LossConfig(use_param_l1=True, weight_param_l1=1.0)
complete_loss = LossConfig(use_param_l1=True, weight_param_l1=1.0,
                           use_mse_curve=True, weight_mse_curve=1.0)

curriculum = LossCurriculumConfig(enabled=True, swap_epoch=5,
                                  warmup=warmup_loss, complete=complete_loss,
                                  reset_lr=False, reset_warmup=False)

criterion = Loss(x_axis, nl, nt, gaussian_cfg, warmup_loss, IdentityNormStats())
warmup    = Warmup(base_cfg, nl, nt)
scheduler = Scheduler([1e-3], warmup, base_cfg, nl, nt)
early     = EarlyStopping(base_cfg, nl, nt)

import torch.nn as nn
opt = torch.optim.AdamW(nn.Linear(2, 2).parameters(), lr=1e-3)

controller = CurriculumController(
    curriculum=curriculum, criterion=criterion, early_stopping=early,
    lr_scheduler=scheduler, warmup=warmup, optimizer=opt,
    update_optimizer=lambda lrs: None, logger=nl,
)

def active_terms(cfg):
    flags = ['use_param_l1', 'use_mse_curve', 'use_cosine_curve', 'use_ssim_curve']
    return [f.replace('use_', '') for f in flags if getattr(cfg, f)]

n_curve_terms = []
swapped_at = None
for epoch in range(12):
    did = controller.maybe_swap(epoch)
    if did:
        swapped_at = epoch
    n_curve_terms.append(len(active_terms(criterion.loss_cfg)))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.step(range(12), n_curve_terms, where='post', color='C0')
ax.axvline(curriculum.swap_epoch, color='red', ls='--', lw=1, label=f'swap epoch = {curriculum.swap_epoch}')
ax.set_xlabel('epoch'); ax.set_ylabel('number of active loss terms')
ax.set_yticks([1, 2])
ax.set_title('Curriculum loss swap')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print('active terms before swap :', active_terms(warmup_loss))
print('active terms after swap  :', active_terms(criterion.loss_cfg))
print('swap fired at epoch      :', swapped_at)


## Expected visual outcome

The overfit collapse reduces a ten-batch loader to a single repeated batch of
the configured size. The checkpoint plot shows the running-best curve as a
non-increasing lower envelope of the noisy validation loss, with green marks
at each improvement. The curriculum plot is a step function that jumps from
one active loss term to two exactly at the swap epoch, confirming the loss
configuration is replaced on schedule.